## Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

import langchain_openrouter
os.environ["OPENROUTER_API_KEY"]= os.getenv("OPENROUTER_API_KEY")

In [6]:
from langchain.chat_models import init_chat_model
model=init_chat_model("openrouter:nvidia/nemotron-3-ultra-550b-a55b:free")
model

ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001A3E0D279B0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={})

In [7]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [8]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatOpenRouter(metadata={'lc_versions': {'langchain-core': '1.7.6', 'langchain': '1.4.9', 'langchain-openrouter': '0.2.9'}}, profile={'name': 'Nemotron 3 Ultra (free)', 'release_date': '2026-06-04', 'last_updated': '2026-06-04', 'open_weights': True, 'max_input_tokens': 1000000, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': False, 'attachment': False, 'temperature': True, 'tool_call_streaming': True}, client=<openrouter.sdk.OpenRouter object at 0x000001A3E0D279B0>, openrouter_api_key=SecretStr('**********'), app_url='https://docs.langchain.com', app_title='LangChain', model_name='nvidia/nemotron-3-ultra-550b-a55b:free', model_kwargs={}), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parame

In [9]:
model.invoke("Provide details about the moview Inception")

AIMessage(content='Here is a comprehensive breakdown of **Christopher Nolan’s *Inception* (2010)**.\n\n---\n\n### **1. Basic Information**\n*   **Director/Writer:** Christopher Nolan\n*   **Producer:** Emma Thomas, Christopher Nolan\n*   **Cinematography:** Wally Pfister\n*   **Score:** Hans Zimmer\n*   **Editor:** Lee Smith\n*   **Release Date:** July 16, 2010\n*   **Runtime:** 148 minutes\n*   **Budget:** ~$160 million\n*   **Box Office:** ~$839 million worldwide\n*   **Genre:** Science Fiction, Action, Heist Thriller, Mystery\n\n---\n\n### **2. Principal Cast & Characters**\n| Actor | Character | Role Description |\n| :--- | :--- | :--- |\n| **Leonardo DiCaprio** | **Dom Cobb** | Protagonist; a skilled "Extractor" haunted by his late wife. |\n| **Marion Cotillard** | **Mal Cobb** | Cobb’s deceased wife; manifests as a violent projection in his subconscious. |\n| **Elliot Page** | **Ariadne** | The "Architect"; designs the dream levels. Named after the Greek myth figure who helped Th

## Message output alongside parsed structure

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'The user is asking for details about the movie "Inception". I need to provide information about this movie. I have a Movie function that can be used to provide movie details. I need to know the year, director, and rating for Inception. I know that Inception was released in 2010, directed by Christopher Nolan. Its rating is typically around 8.8/10 on IMDb. I\'ll use the Movie function to provide these details.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user is asking for details about the movie "Inception". I need to provide information about this movie. I have a Movie function that can be used to provide movie details. I need to know the year, director, and rating for Inception. I know that Inception was released in 2010, directed by Christopher Nolan. Its rating is typically around 8.8/10 on IMDb. I\'ll use the Movie function to provide these details.'}]}, respo

## Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Marion Cotillard', role='Mal Cobb'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Michael Caine', role='Professor Stephen Miles'), Actor(name='Joseph Gordon-Levitt', role='Arthur')], genres=['Science Fiction', 'Action', 'Thriller', 'Heist'], budget=160.0)

## TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [12]:
from typing_extensions import TypedDict,Annotated

class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_withtypedict=model.with_structured_output(MovieDict)
response=model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'director': 'Joss Whedon',
 'rating': 8.0}

In [13]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Avengers")
response

{'title': 'The Avengers',
 'year': 2012,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Clark Gregg', 'role': 'Phil Coulson'},
  {'name': 'Cobie Smulders', 'role': 'Maria Hill'},
  {'name': 'Stellan Skarsgård', 'role': 'Erik Selvig'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi'],
 'budget': 220000000}

In [14]:
model.profile

{'name': 'Nemotron 3 Ultra (free)',
 'release_date': '2026-06-04',
 'last_updated': '2026-06-04',
 'open_weights': True,
 'max_input_tokens': 1000000,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': False,
 'attachment': False,
 'temperature': True,
 'tool_call_streaming': True}

## DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [15]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="openrouter:nvidia/nemotron-3-ultra-550b-a55b:free",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='cf449f7e-049d-4838-9973-d9f23ad2d69b'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user wants me to extract contact info from the given string: "John Doe, john@example.com, (555) 123-4567". I need to parse this into name, email, and phone number. The string appears to be in the format: name, email, phone. I can split by commas. Let\'s do that: "John Doe", "john@example.com", "(555) 123-4567". That seems straightforward. I\'ll use the ContactInfo function with these values.', 'reasoning_details': [{'type': 'reasoning.text', 'format': 'unknown', 'index': 0, 'text': 'The user wants me to extract contact info from the given string: "John Doe, john@example.com, (555) 123-4567". I need to parse this into name, email, and phone number. The string appears to be in the format: name, email, phone. I can split by comm

In [16]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person

agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# {'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str # The name of the person
    email: str # The email address of the person
    phone: str # The phone number of the person


agent = create_agent(
    model="gpt-5",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]
# ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')